# Indicators Analysis and Prediction
### T10YIEM · FEDFUNDS · MSPUS · DTWEXBGS · Multi-Factor Ridge Regression

**Setup:** Place all CSV files inside a folder called `Data/` in the same directory as this notebook.

Required files:
- `Data/T10YIEM.csv`
- `Data/FEDFUNDS.csv`
- `Data/MSPUS.csv`
- `Data/DTWEXBGS.csv`

---

## Analytical Approach & Structure

This notebook is structured in two phases. **Phase 1 (Parts 1–4)** performs a standalone exploratory data analysis (EDA) and univariate forecast for each economic indicator individually. **Phase 2 (Part 5)** combines all four series into a multi-factor regression model to forecast inflation expectations.

The rationale for analysing each series independently before combining them is deliberate:

1. **Understanding each series on its own terms first** avoids misinterpreting multivariate results. For example, knowing that the Fed Funds Rate is non-stationary (a unit root process) before building a joint model informs how lags and differences should be constructed in Part 5.
2. **Univariate forecasts serve as benchmarks.** If the multi-factor model in Part 5 does not outperform a simple Holt-Winters forecast, that is a meaningful finding — it suggests the macro relationships are weaker than expected over the forecast horizon.
3. **The ordering of series reflects economic causality.** Inflation expectations (T10YIEM) are the target variable and are examined first. The Fed Funds Rate is examined second because monetary policy is the primary tool used to influence inflation. Housing and the dollar index are examined third and fourth as secondary transmission channels.

Within each Part, the same analytical sequence is followed: load → visualise → distribution → rolling statistics → stationarity test → autocorrelation → forecast. This consistent structure makes it straightforward to compare behaviour across series.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# ── All CSVs are read from this folder ──────────────────────────────────────
DATA_DIR = 'Data/'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print(f'Reading data from: {DATA_DIR}')

---
# Part 1 — Inflation Expectations (T10YIEM)

The 10-Year Breakeven Inflation Rate is the market's implied expectation of average inflation over the next decade, derived from the spread between nominal and inflation-protected Treasury yields. It is the **target variable** for the multi-factor model in Part 5, and is therefore analysed first.

This series is examined before the predictors (Fed Funds Rate, housing, dollar index) to establish a clear picture of what the model is trying to explain — its historical range, volatility regime, and autocorrelation structure — before any relationships with other variables are introduced.

---

### 1.1 Load & Inspect

Basic shape, summary statistics, and missing value check. This step confirms the date range and frequency of the data before any analysis begins, and catches any data quality issues early. Identifying the frequency (monthly here) is important because it determines the lag structure used in later steps — a 12-period lag corresponds to one year for monthly data.

In [ ]:
df_inf = pd.read_csv(DATA_DIR + 'T10YIEM.csv', parse_dates=['observation_date'])
df_inf.rename(columns={'observation_date': 'date', 'T10YIEM': 'inflation'}, inplace=True)
df_inf.set_index('date', inplace=True)
df_inf.sort_index(inplace=True)

print(df_inf.shape)
print(df_inf.describe())
print(f'Missing values: {df_inf.inflation.isna().sum()}')

### 1.2 Full Time Series

Plotting the full time series as the first visual step establishes the overall trend, identifies structural breaks, and reveals any obvious outliers or data anomalies. The mean is overlaid as a reference line to show whether the series spends most of its time above or below its long-run average — relevant for understanding whether recent values represent a reversion to mean or a new regime.

Looking at the raw series before any transformations preserves interpretability and prevents missing patterns that differencing or smoothing would obscure.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_inf.index, df_inf['inflation'], color='steelblue', linewidth=1.2)
ax.axhline(df_inf['inflation'].mean(), color='red', linestyle='--',
           label=f'Mean: {df_inf["inflation"].mean():.2f}%')
ax.set_title('10-Year Breakeven Inflation Rate (T10YIEM)')
ax.set_ylabel('Inflation Expectation (%)')
ax.legend()
plt.tight_layout()
plt.show()

### 1.3 Distribution & Rolling Stats

The histogram reveals whether the series follows a roughly normal distribution or is skewed — relevant because OLS regression assumes normally distributed residuals. A bimodal or heavily skewed distribution can indicate the presence of distinct regimes (e.g. pre- and post-COVID inflation environments) that a single linear model may not handle well.

The 12-month rolling mean and ±1 standard deviation band are calculated to show how the central tendency and volatility of the series have evolved over time. A widening band indicates periods of higher uncertainty, while a narrowing band suggests a stable regime. This is more informative than a static standard deviation because economic volatility is not constant.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_inf['inflation'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Inflation Expectations')
axes[0].set_xlabel('%')

roll = df_inf['inflation'].rolling(12)
axes[1].plot(df_inf.index, df_inf['inflation'], alpha=0.4, label='Monthly')
axes[1].plot(df_inf.index, roll.mean(), label='12M Rolling Mean', linewidth=2)
axes[1].fill_between(df_inf.index, roll.mean() - roll.std(), roll.mean() + roll.std(),
                     alpha=0.2, label='±1 Std Dev')
axes[1].set_title('Rolling Mean ± Std Dev')
axes[1].legend()
plt.tight_layout()
plt.show()

### 1.4 YoY Change & Stationarity Test

Year-over-year changes are computed to show the rate of acceleration or deceleration in the series, independent of its absolute level. This is particularly useful for detecting turning points — a series can be at a high absolute level but decelerating, which has different implications than a series that is both high and accelerating.

The **Augmented Dickey-Fuller (ADF) test** is run to formally test for stationarity. This matters for modelling because a non-stationary series (one with a unit root) has statistical properties that change over time — specifically, its mean and variance are not constant. Feeding non-stationary series directly into a regression without adjustment can produce spurious correlations. The ADF result informs whether differencing or lagging is required in the feature engineering step of Part 5.

The bar chart uses a two-colour scheme (blue for positive, red for negative) to make periods of acceleration and deceleration immediately visible.

In [ ]:
df_inf['yoy_change'] = df_inf['inflation'].diff(12)
fig, ax = plt.subplots()
ax.bar(df_inf.index, df_inf['yoy_change'],
       color=np.where(df_inf['yoy_change'] >= 0, 'steelblue', 'tomato'), width=20)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Year-over-Year Change in Inflation Expectations')
ax.set_ylabel('Change (pp)')
plt.tight_layout()
plt.show()

adf = adfuller(df_inf['inflation'].dropna())
print(f'ADF Statistic: {adf[0]:.4f}, p-value: {adf[1]:.4f}')
print('Series is', 'stationary' if adf[1] < 0.05 else 'non-stationary')

### 1.5 ACF / PACF

The **Autocorrelation Function (ACF)** measures how strongly the series correlates with its own past values at increasing lags. Slow decay in the ACF is further evidence of non-stationarity. The **Partial ACF (PACF)** isolates the direct effect of each lag, removing the indirect influence of intervening lags.

Together, these plots are used to determine the appropriate lag structure for time series models. Significant PACF spikes at specific lags indicate which historical values carry the most predictive information — this directly informs which lag features (lag 1, 3, 6, 12) are constructed in the feature engineering step of Part 5. A spike at lag 12 for monthly data, for example, indicates strong seasonality or annual cyclicality.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_inf['inflation'].dropna(), lags=36, ax=axes[0], title='ACF')
plot_pacf(df_inf['inflation'].dropna(), lags=36, ax=axes[1], title='PACF')
plt.tight_layout()
plt.show()

### 1.6 Forecast: Linear, Polynomial & Holt-Winters (36 months)

Three univariate forecasting methods are applied and compared as a baseline before introducing the multi-factor model in Part 5:

- **Linear Regression on time index:** The simplest possible model — it fits a straight line through the historical series and extends it. If the series has a clear long-run trend, this works well. It is included as the baseline benchmark that any more complex model should outperform.
- **Polynomial Regression (degree 3):** Allows for curvature in the trend, capturing acceleration and deceleration. Degree 3 is chosen as a balance — degree 1 is too rigid, while higher degrees tend to overfit at the edges of the data and produce unrealistic extrapolations.
- **Holt-Winters Exponential Smoothing:** A well-established method for series with both trend and seasonality. Unlike regression, it weights recent observations more heavily than older ones, making it more responsive to recent regime changes. It is the most practically grounded of the three for a 36-month horizon.

Comparing all three on the same chart makes divergence between methods visible — wide divergence signals high forecast uncertainty.

In [ ]:
series = df_inf['inflation'].dropna()
t = np.arange(len(series)).reshape(-1, 1)
y = series.values
future_steps = 36
t_future = np.arange(len(series), len(series) + future_steps).reshape(-1, 1)
future_dates = pd.date_range(series.index[-1] + pd.DateOffset(months=1), periods=future_steps, freq='MS')

lr = LinearRegression().fit(t, y)
lr_pred = lr.predict(t_future)
print(f'Linear R²: {r2_score(y, lr.predict(t)):.4f}')

poly = make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(t, y)
poly_pred = poly.predict(t_future)
print(f'Polynomial (deg 3) R²: {r2_score(y, poly.predict(t)):.4f}')

hw_model = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
hw_pred = hw_model.forecast(future_steps)
print(f'Holt-Winters AIC: {hw_model.aic:.2f}')

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(series.index, series, color='black', linewidth=1, label='Observed', alpha=0.7)
ax.plot(future_dates, lr_pred, '--', color='royalblue', linewidth=2, label='Linear Regression')
ax.plot(future_dates, poly_pred, '--', color='darkorange', linewidth=2, label='Polynomial (deg 3)')
ax.plot(future_dates, hw_pred.values, '--', color='green', linewidth=2, label='Holt-Winters')
ax.axvline(series.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
ax.set_title('Inflation Expectation Forecast — Next 36 Months')
ax.set_ylabel('Inflation Expectation (%)')
ax.legend()
plt.tight_layout()
plt.show()

---
# Part 2 — Federal Funds Rate (FEDFUNDS)

The Federal Funds Rate is the interest rate at which banks lend to each other overnight, set by the Federal Reserve as its primary monetary policy tool. It is analysed second — immediately after the inflation target — because it is the most direct policy lever used to influence inflation. The relationship is well-established: rate hikes are intended to cool inflation by tightening credit conditions; rate cuts are used to stimulate activity during periods of low inflation or recession.

The Fed Funds Rate is examined as a predictor before housing and the dollar index because its transmission to inflation is more direct and faster-acting. The housing market and exchange rate respond to interest rate changes, so in some respects they are downstream of this variable.

---

### 2.1 Load & Inspect

The FEDFUNDS series runs from 1954, making it one of the longest series in this dataset. The full history is loaded and retained for context, but the forecast model in section 2.7 is deliberately fitted on data from 2000 onwards — the post-Volcker era — because the monetary policy regime of the 1970s and 1980s (characterised by double-digit rates) is not representative of the current environment and would bias trend extrapolations upward.

In [ ]:
df_ff = pd.read_csv(DATA_DIR + 'FEDFUNDS.csv', parse_dates=['observation_date'])
df_ff.rename(columns={'observation_date': 'date', 'FEDFUNDS': 'rate'}, inplace=True)
df_ff.set_index('date', inplace=True)
df_ff.sort_index(inplace=True)

print(df_ff.shape)
print(df_ff.describe())
print(f'Missing values: {df_ff.rate.isna().sum()}')

### 2.2 Full Time Series

Recession periods are shaded to contextualise the rate cycle — the Fed typically cuts rates aggressively during recessions, producing the characteristic sharp drops visible in the chart. These shading bands provide immediate visual context for the structural breaks in the series, making it easier to interpret the distribution and rolling statistics that follow.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_ff.index, df_ff['rate'], color='steelblue', linewidth=1.2)
ax.axhline(df_ff['rate'].mean(), color='red', linestyle='--',
           label=f'Mean: {df_ff["rate"].mean():.2f}%')
recessions = [('1973-11-01','1975-03-01'),('1980-01-01','1982-11-01'),
              ('2007-12-01','2009-06-01'),('2020-02-01','2020-04-01')]
for s, e in recessions:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.12, color='red')
ax.set_title('Federal Funds Effective Rate')
ax.set_ylabel('Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Distribution & Rolling Stats

The distribution of the Fed Funds Rate is expected to be multimodal, reflecting distinct policy regimes across the full sample (high-rate 1970s–80s, low-rate 2010s, current elevated cycle). This multimodality is a key reason why the Holt-Winters model in section 2.7 is fitted on post-2000 data rather than the full history — fitting on the full distribution would produce a trend estimate anchored to an unrepresentative average.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_ff['rate'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Fed Funds Rate')
axes[0].set_xlabel('%')

roll = df_ff['rate'].rolling(12)
axes[1].plot(df_ff.index, df_ff['rate'], alpha=0.4, label='Monthly')
axes[1].plot(df_ff.index, roll.mean(), label='12M Rolling Mean', linewidth=2)
axes[1].fill_between(df_ff.index, roll.mean() - roll.std(), roll.mean() + roll.std(),
                     alpha=0.2, label='±1 Std Dev')
axes[1].set_title('Rolling Mean ± Std Dev')
axes[1].legend()
plt.tight_layout()
plt.show()

### 2.4 YoY Change & Stationarity Test

Year-over-year changes in the Fed Funds Rate represent the *pace* of monetary tightening or easing, which is often more relevant to inflation dynamics than the absolute level. A rapid series of rate hikes (large positive YoY change) has a different economic effect than a single large hike from the same absolute level.

The ADF test result for this series is particularly important: if the rate is non-stationary over the full sample, this suggests that using the raw level as a predictor in Part 5 could introduce spurious relationships. Both the level and the 12-month change are therefore included as separate features in the Part 5 feature engineering step.

In [ ]:
df_ff['yoy_change'] = df_ff['rate'].diff(12)
fig, ax = plt.subplots()
ax.bar(df_ff.index, df_ff['yoy_change'],
       color=np.where(df_ff['yoy_change'] >= 0, 'steelblue', 'tomato'), width=20)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Year-over-Year Change in Fed Funds Rate')
ax.set_ylabel('Change (pp)')
plt.tight_layout()
plt.show()

adf = adfuller(df_ff['rate'].dropna())
print(f'ADF Statistic: {adf[0]:.4f}, p-value: {adf[1]:.4f}')
print('Series is', 'stationary' if adf[1] < 0.05 else 'non-stationary')

### 2.5 ACF / PACF

For the Fed Funds Rate, the ACF is expected to show very slow decay — rates are highly persistent, changing only at scheduled FOMC meetings and often in gradual steps. This persistence means short lags (lag 1, lag 3) will be highly significant predictors, while longer lags (lag 12) capture the annual policy cycle. These observations directly inform the lag selection in Part 5.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_ff['rate'].dropna(), lags=48, ax=axes[0], title='ACF')
plot_pacf(df_ff['rate'].dropna(), lags=48, ax=axes[1], title='PACF')
plt.tight_layout()
plt.show()

### 2.6 Recent Cycle (2015–Present)

The post-2015 period is isolated to examine the two most recent full rate cycles: the 2015–2018 normalisation, the 2020 emergency cut to zero, and the 2022–2023 tightening cycle — the most aggressive since the Volcker era. This zoom is included because the forecast in section 2.7 is most influenced by the most recent trend, and reviewing this period separately ensures the forecast starting conditions are clearly understood before the projections are generated.

In [ ]:
recent = df_ff['rate']['2015':]
fig, ax = plt.subplots()
ax.plot(recent.index, recent, color='steelblue', linewidth=1.5)
ax.set_title('Fed Funds Rate — Recent Tightening Cycle (2015–Present)')
ax.set_ylabel('Rate (%)')
plt.tight_layout()
plt.show()

### 2.7 Forecast: Linear, Polynomial & Holt-Winters (36 months)

The Holt-Winters model is fitted on data from 2000 onwards rather than the full history. This is a deliberate choice: the 1970s–80s rate environment (peak rates above 19%) is structurally different from the modern era and would pull the model's trend estimate significantly upward, producing unrealistic forecasts. Using a regime-appropriate training window produces a more credible baseline.

Forecast values are clipped at zero — negative nominal policy rates are theoretically possible but have never been implemented in the US, making negative projections unreliable as planning inputs.

The dual-panel chart (full history + zoomed last 5 years) is used here and throughout because the full history provides context for long-run positioning while the zoom panel makes the forecast trajectory easier to read at the relevant scale.

In [ ]:
series = df_ff['rate'].dropna()
t = np.arange(len(series)).reshape(-1, 1)
y = series.values
future_steps = 36
t_future = np.arange(len(series), len(series) + future_steps).reshape(-1, 1)
future_dates = pd.date_range(series.index[-1] + pd.DateOffset(months=1), periods=future_steps, freq='MS')

lr = LinearRegression().fit(t, y)
lr_pred_ff = lr.predict(t_future)
print(f'Linear R²: {r2_score(y, lr.predict(t)):.4f}')

poly = make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(t, y)
poly_pred_ff = poly.predict(t_future)
print(f'Polynomial (deg 3) R²: {r2_score(y, poly.predict(t)):.4f}')

series_recent = series['2000':]
hw_model = ExponentialSmoothing(series_recent, trend='add', seasonal='add', seasonal_periods=12).fit()
hw_pred_ff = hw_model.forecast(future_steps).clip(lower=0)
print(f'Holt-Winters AIC: {hw_model.aic:.2f}')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
axes[0].plot(series.index, series, color='black', linewidth=1, alpha=0.6, label='Observed')
axes[0].plot(future_dates, lr_pred_ff, '--', color='royalblue', linewidth=2, label='Linear Regression')
axes[0].plot(future_dates, poly_pred_ff, '--', color='darkorange', linewidth=2, label='Polynomial (deg 3)')
axes[0].plot(future_dates, hw_pred_ff.values, '--', color='green', linewidth=2, label='Holt-Winters (2000+ fit)')
axes[0].axvline(series.index[-1], color='gray', linestyle=':', linewidth=1.5)
axes[0].set_title('Federal Funds Rate — Full History + 36M Forecast')
axes[0].set_ylabel('Rate (%)')
axes[0].legend()

zoom_start = series.index[-1] - pd.DateOffset(years=5)
axes[1].plot(series[zoom_start:].index, series[zoom_start:], color='black', linewidth=1.5, alpha=0.7, label='Observed')
axes[1].plot(future_dates, lr_pred_ff, '--', color='royalblue', linewidth=2, label='Linear Regression')
axes[1].plot(future_dates, poly_pred_ff, '--', color='darkorange', linewidth=2, label='Polynomial (deg 3)')
axes[1].plot(future_dates, hw_pred_ff.values, '--', color='green', linewidth=2, label='Holt-Winters')
axes[1].axvline(series.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
axes[1].set_title('Zoomed: Last 5 Years + 36M Forecast')
axes[1].set_ylabel('Rate (%)')
axes[1].legend()
plt.tight_layout()
plt.show()

---
# Part 3 — Median Home Sale Price (MSPUS)

The median US home sale price is examined third because housing represents the largest single component of household expenditure and is a key channel through which monetary policy transmits to the real economy — rate hikes increase mortgage costs, which should cool demand and put downward pressure on prices. Housing is also a significant contributor to shelter inflation in the CPI, creating a direct link to the inflation target variable.

This series is quarterly, unlike the monthly frequency of Parts 1, 2, and 4. This difference in frequency is handled in Part 5 by forward-filling quarterly values to monthly — a standard approach that preserves the most recent known value rather than interpolating, which would introduce information from the future.

---

### 3.1 Load & Inspect

The quarterly frequency and date range are noted explicitly here because they differ from the other series. Confirming the exact start and end dates is important before the series is merged with monthly data in Part 5 — a mismatch in the time index is a common source of silent errors in multi-series analysis.

In [ ]:
df_hs = pd.read_csv(DATA_DIR + 'MSPUS.csv', parse_dates=['observation_date'])
df_hs.rename(columns={'observation_date': 'date', 'MSPUS': 'price'}, inplace=True)
df_hs.set_index('date', inplace=True)
df_hs.sort_index(inplace=True)

print(df_hs.shape)
print(df_hs.describe())
print(f'Missing values: {df_hs.price.isna().sum()}')
print(f'Frequency: Quarterly | Range: {df_hs.index[0].date()} to {df_hs.index[-1].date()}')

### 3.2 Full Time Series

The housing price series is plotted in thousands of dollars rather than raw values for readability. The GFC (2007–2009) and COVID (2020) recession periods are shaded because housing was the *cause* of the 2008 crisis and a significant beneficiary of pandemic-era fiscal and monetary stimulus — both events represent structural breaks that are highly visible in this series and are important context for interpreting the forecast.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_hs.index, df_hs['price'] / 1000, color='goldenrod', linewidth=1.5)
ax.axhline(df_hs['price'].mean() / 1000, color='red', linestyle='--',
           label=f'Mean: ${df_hs["price"].mean()/1000:.0f}K')
for s, e in [('2007-12-01','2009-06-01'),('2020-02-01','2020-04-01')]:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.15, color='red')
ax.set_title('Median US Home Sale Price (MSPUS)')
ax.set_ylabel('Price ($K)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 3.3 Distribution & Rolling Stats

A 4-quarter (1-year) rolling window is used rather than the 12-month window used for monthly series — this is the equivalent periodicity for quarterly data. The distribution of home prices is expected to be right-skewed given the sharp post-COVID appreciation, and understanding this skew is relevant when interpreting regression coefficients in Part 5, where the raw price level is used as a predictor.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_hs['price'].dropna() / 1000, bins=40, color='goldenrod', edgecolor='white')
axes[0].set_title('Distribution of Median Home Price')
axes[0].set_xlabel('Price ($K)')

roll = df_hs['price'].rolling(4)
axes[1].plot(df_hs.index, df_hs['price'] / 1000, alpha=0.4, label='Quarterly')
axes[1].plot(df_hs.index, roll.mean() / 1000, label='4Q Rolling Mean', linewidth=2)
axes[1].fill_between(df_hs.index,
                     (roll.mean() - roll.std()) / 1000,
                     (roll.mean() + roll.std()) / 1000,
                     alpha=0.2, label='±1 Std Dev')
axes[1].set_title('Rolling Mean ± Std Dev')
axes[1].set_ylabel('Price ($K)')
axes[1].legend()
plt.tight_layout()
plt.show()

### 3.4 YoY Change & Stationarity Test

Both the dollar change and the percentage change are plotted for housing. The dollar change shows the absolute acceleration in price growth, while the percentage change normalises for the price level — both are informative. A 10% YoY increase at a median price of $200K means something very different to the same rate at $450K in terms of affordability and demand sustainability.

The ADF test for housing prices is expected to confirm non-stationarity — prices have trended strongly upward over the full sample. The percentage change and lagged levels are therefore the more appropriate features to include in the regression, which is reflected in the YoY change features constructed in Part 5.

In [ ]:
df_hs['yoy_change'] = df_hs['price'].diff(4)
df_hs['yoy_pct'] = df_hs['price'].pct_change(4) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(df_hs.index, df_hs['yoy_change'] / 1000,
            color=np.where(df_hs['yoy_change'] >= 0, 'steelblue', 'tomato'), width=60)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('YoY Change in Median Home Price ($K)')

axes[1].bar(df_hs.index, df_hs['yoy_pct'],
            color=np.where(df_hs['yoy_pct'] >= 0, 'steelblue', 'tomato'), width=60)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('YoY % Change in Median Home Price')
axes[1].set_ylabel('%')
plt.tight_layout()
plt.show()

adf = adfuller(df_hs['price'].dropna())
print(f'ADF Statistic: {adf[0]:.4f}, p-value: {adf[1]:.4f}')
print('Series is', 'stationary' if adf[1] < 0.05 else 'non-stationary')

### 3.5 ACF / PACF

Housing prices are highly persistent — once prices start rising they tend to continue rising for multiple quarters before reversing. This is captured by the ACF, which is expected to show very slow decay. The PACF reveals whether this persistence is primarily driven by the most recent quarter (lag 1) or carries significant information from further back, which informs lag selection in Part 5.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_hs['price'].dropna(), lags=24, ax=axes[0], title='ACF')
plot_pacf(df_hs['price'].dropna(), lags=24, ax=axes[1], title='PACF')
plt.tight_layout()
plt.show()

### 3.6 Recent Cycle (2010–Present)

The post-GFC period is isolated to focus on the recovery and subsequent boom. This zoom serves a specific purpose: the 2012+ period is also the training window used for the Holt-Winters model in the next section, so reviewing this period first allows the starting conditions of the forecast to be assessed visually before the projections are generated. The post-2012 window is chosen because it excludes the GFC trough, which would otherwise pull the model's level and trend estimates downward.

In [ ]:
recent = df_hs['price']['2010':]
fig, ax = plt.subplots()
ax.plot(recent.index, recent / 1000, color='goldenrod', linewidth=1.8)
ax.set_title('Median Home Price — Post-GFC Recovery (2010–Present)')
ax.set_ylabel('Price ($K)')
plt.tight_layout()
plt.show()

### 3.7 Forecast: Linear, Polynomial & Holt-Winters (12 quarters / 3 years)

The forecast horizon is expressed in quarters (12 quarters = 3 years) to match the native frequency of this series. The Holt-Winters model uses a seasonal period of 4 (quarters per year) rather than 12 — this is the equivalent annual cycle for quarterly data.

The Holt-Winters model is fitted from 2012 onwards, excluding the GFC crash and recovery period. This is because the 2008–2012 period represents an extreme structural break — a once-in-a-generation housing collapse — that is not representative of the current market structure and would cause the model to underestimate the trend if included.

In [ ]:
series = df_hs['price'].dropna()
t = np.arange(len(series)).reshape(-1, 1)
y = series.values
future_steps = 12
t_future = np.arange(len(series), len(series) + future_steps).reshape(-1, 1)
future_dates = pd.date_range(series.index[-1] + pd.DateOffset(months=3), periods=future_steps, freq='QS')

lr = LinearRegression().fit(t, y)
lr_pred_hs = lr.predict(t_future)
print(f'Linear R²: {r2_score(y, lr.predict(t)):.4f}')

poly = make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(t, y)
poly_pred_hs = poly.predict(t_future)
print(f'Polynomial (deg 3) R²: {r2_score(y, poly.predict(t)):.4f}')

series_recent = series['2012':]
hw_model = ExponentialSmoothing(series_recent, trend='add', seasonal='add', seasonal_periods=4).fit()
hw_pred_hs = hw_model.forecast(future_steps)
print(f'Holt-Winters AIC: {hw_model.aic:.2f}')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
axes[0].plot(series.index, series / 1000, color='black', linewidth=1, alpha=0.6, label='Observed')
axes[0].plot(future_dates, lr_pred_hs / 1000, '--', color='royalblue', linewidth=2, label='Linear Regression')
axes[0].plot(future_dates, poly_pred_hs / 1000, '--', color='darkorange', linewidth=2, label='Polynomial (deg 3)')
axes[0].plot(future_dates, hw_pred_hs.values / 1000, '--', color='green', linewidth=2, label='Holt-Winters (2012+ fit)')
axes[0].axvline(series.index[-1], color='gray', linestyle=':', linewidth=1.5)
axes[0].set_title('Median US Home Sale Price — Full History + 3Y Forecast')
axes[0].set_ylabel('Price ($K)')
axes[0].legend()

zoom_start = series.index[-1] - pd.DateOffset(years=5)
axes[1].plot(series[zoom_start:].index, series[zoom_start:] / 1000,
             color='black', linewidth=1.5, alpha=0.7, label='Observed')
axes[1].plot(future_dates, lr_pred_hs / 1000, '--', color='royalblue', linewidth=2, label='Linear Regression')
axes[1].plot(future_dates, poly_pred_hs / 1000, '--', color='darkorange', linewidth=2, label='Polynomial (deg 3)')
axes[1].plot(future_dates, hw_pred_hs.values / 1000, '--', color='green', linewidth=2, label='Holt-Winters')
axes[1].axvline(series.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
axes[1].set_title('Zoomed: Last 5 Years + 3Y Forecast')
axes[1].set_ylabel('Price ($K)')
axes[1].legend()
plt.tight_layout()
plt.show()

print('\n--- 12-Quarter Forecast Summary ---')
summary = pd.DataFrame({
    'Quarter': future_dates.strftime('%Y-Q') + ((future_dates.month - 1) // 3 + 1).astype(str),
    'Linear ($K)': (lr_pred_hs / 1000).round(1),
    'Polynomial ($K)': (poly_pred_hs / 1000).round(1),
    'Holt-Winters ($K)': (hw_pred_hs.values / 1000).round(1)
})
print(summary.to_string(index=False))

---
# Part 4 — Broad Dollar Index (DTWEXBGS)

The Broad Dollar Index measures the value of the US dollar against a trade-weighted basket of foreign currencies. It is examined last among the individual series because its relationship with inflation operates through an indirect channel: a stronger dollar makes imports cheaper, reducing cost-push inflation, while a weaker dollar raises import prices. This is a slower-acting transmission mechanism than the Fed Funds Rate, which is why the dollar index is positioned as a secondary predictor in the multi-factor model.

This series is daily — the highest frequency in this dataset. It is resampled to monthly averages throughout this analysis and in Part 5, both for consistency with the other series and because daily fluctuations in the dollar index are largely noise from the perspective of monthly inflation expectations.

---

### 4.1 Load & Inspect

The daily data is loaded and a monthly-resampled version is created immediately. Both are retained: the daily series is used for volatility analysis (section 4.4, which requires daily granularity), while the monthly series is used for all trend analysis and forecasting. This split is made explicit here so that it is clear which series feeds into each subsequent step.

In [ ]:
df_dxy = pd.read_csv(DATA_DIR + 'DTWEXBGS.csv', parse_dates=['observation_date'])
df_dxy.rename(columns={'observation_date': 'date', 'DTWEXBGS': 'dxy'}, inplace=True)
df_dxy.set_index('date', inplace=True)
df_dxy.sort_index(inplace=True)
df_dxy.dropna(inplace=True)
df_dxy_monthly = df_dxy.resample('MS').mean()

print(df_dxy.shape)
print(df_dxy.describe())
print(f'Frequency: Daily | Range: {df_dxy.index[0].date()} to {df_dxy.index[-1].date()}')

### 4.2 Full Time Series

Both the raw daily series and the monthly average are plotted together. The daily series provides a sense of day-to-day volatility, while the monthly average reveals the underlying trend without the noise. Key macro events are shaded — the GFC flight to safety, the 2014–2016 dollar rally driven by Fed taper expectations, COVID, and the 2022 Fed hike cycle — because these events explain the largest movements in the series and are relevant context when interpreting the forecast.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_dxy.index, df_dxy['dxy'], color='purple', linewidth=0.8, alpha=0.5, label='Daily')
ax.plot(df_dxy_monthly.index, df_dxy_monthly['dxy'], color='indigo', linewidth=1.5, label='Monthly Avg')
ax.axhline(df_dxy['dxy'].mean(), color='red', linestyle='--',
           label=f'Mean: {df_dxy["dxy"].mean():.2f}')
for s, e in [('2008-09-01','2009-03-01'),('2020-02-01','2020-04-01'),('2022-01-01','2022-11-01')]:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.1, color='orange')
ax.set_title('USD Broad Dollar Index (DTWEXBGS)')
ax.set_ylabel('Index Value')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 4.3 Distribution & Rolling Stats

The distribution analysis is performed on the daily series to capture the full range of values, while the rolling statistics are computed on the monthly series. This distinction matters: the daily histogram reflects the full empirical distribution including short-term spikes, while the monthly rolling statistics smooth these out to show the trend in central tendency. Using the daily series for the histogram gives a more accurate picture of the true distribution shape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_dxy['dxy'].dropna(), bins=60, color='purple', edgecolor='white', alpha=0.8)
axes[0].axvline(df_dxy['dxy'].mean(), color='red', linestyle='--',
                label=f'Mean: {df_dxy["dxy"].mean():.1f}')
axes[0].set_title('Distribution of Dollar Index (Daily)')
axes[0].legend()

roll = df_dxy_monthly['dxy'].rolling(12)
axes[1].plot(df_dxy_monthly.index, df_dxy_monthly['dxy'], alpha=0.5, label='Monthly Avg')
axes[1].plot(df_dxy_monthly.index, roll.mean(), label='12M Rolling Mean', linewidth=2)
axes[1].fill_between(df_dxy_monthly.index,
                     roll.mean() - roll.std(), roll.mean() + roll.std(),
                     alpha=0.2, label='±1 Std Dev')
axes[1].set_title('12M Rolling Mean ± Std Dev (Monthly)')
axes[1].legend()
plt.tight_layout()
plt.show()

### 4.4 Daily Returns & Volatility

This section is specific to the dollar index and is not replicated for the other series. Daily returns and a 30-day rolling volatility measure are included here because the dollar index is a financial market series — unlike the Fed Funds Rate (a policy variable) or housing prices (a slow-moving quarterly series), it responds immediately to news and can exhibit clustering of volatility. Understanding this volatility structure is relevant for interpreting how quickly dollar movements feed through to inflation expectations.

The 30-day window for rolling volatility is chosen because it corresponds roughly to one calendar month, making it directly comparable to the monthly aggregation used elsewhere.

In [ ]:
df_dxy['daily_return'] = df_dxy['dxy'].pct_change() * 100
df_dxy['roll_vol_30d'] = df_dxy['daily_return'].rolling(30).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df_dxy.index, df_dxy['daily_return'], color='purple', linewidth=0.5, alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Daily Returns (%)')
axes[0].set_ylabel('%')
axes[1].plot(df_dxy.index, df_dxy['roll_vol_30d'], color='darkorange', linewidth=1)
axes[1].set_title('30-Day Rolling Volatility')
axes[1].set_ylabel('Std Dev (%)')
plt.tight_layout()
plt.show()

### 4.5 YoY Change & Stationarity Test

YoY changes are computed on the monthly series rather than the daily series, consistent with the rest of the analysis. This ensures that the stationarity test and YoY change visualisation are directly comparable to those produced for the other series in Parts 1–3, making cross-series comparisons valid.

In [ ]:
df_dxy_monthly['yoy_change'] = df_dxy_monthly['dxy'].diff(12)
df_dxy_monthly['yoy_pct']    = df_dxy_monthly['dxy'].pct_change(12) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(df_dxy_monthly.index, df_dxy_monthly['yoy_change'],
            color=np.where(df_dxy_monthly['yoy_change'] >= 0, 'steelblue', 'tomato'), width=20)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('YoY Change in Dollar Index')
axes[1].bar(df_dxy_monthly.index, df_dxy_monthly['yoy_pct'],
            color=np.where(df_dxy_monthly['yoy_pct'] >= 0, 'steelblue', 'tomato'), width=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('YoY % Change in Dollar Index')
axes[1].set_ylabel('%')
plt.tight_layout()
plt.show()

adf = adfuller(df_dxy_monthly['dxy'].dropna())
print(f'ADF Statistic: {adf[0]:.4f}, p-value: {adf[1]:.4f}')
print('Series is', 'stationary' if adf[1] < 0.05 else 'non-stationary')

### 4.6 ACF / PACF

The ACF and PACF are computed on the monthly series. Given the broadly upward trend in the dollar index since 2014, the ACF is expected to show very slow decay — indicative of a non-stationary, trending series. This is consistent with the expectation from section 4.5, and confirms that YoY changes rather than levels are the more appropriate features for the regression in Part 5.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_dxy_monthly['dxy'].dropna(), lags=36, ax=axes[0], title='ACF (Monthly Avg)')
plot_pacf(df_dxy_monthly['dxy'].dropna(), lags=36, ax=axes[1], title='PACF (Monthly Avg)')
plt.tight_layout()
plt.show()

### 4.7 Forecast: Linear, Polynomial & Holt-Winters (36 months)

The dollar index forecast uses the full monthly history rather than a truncated window, unlike the Fed Funds Rate model. This is because the dollar index does not have the same structural regime break problem as the Fed Funds Rate — while there are periods of significant movement, there is no single dominant structural break (like the Volcker era) that would make pre-break data unrepresentative. The full history provides more data for trend estimation without introducing regime distortion.

*Note: The dollar index is driven by macro policy differentials, trade flows, and geopolitical factors. These models extrapolate statistical trends and should not be interpreted as fundamental exchange rate forecasts.*

In [ ]:
series = df_dxy_monthly['dxy'].dropna()
t = np.arange(len(series)).reshape(-1, 1)
y = series.values
future_steps = 36
t_future = np.arange(len(series), len(series) + future_steps).reshape(-1, 1)
future_dates = pd.date_range(series.index[-1] + pd.DateOffset(months=1), periods=future_steps, freq='MS')

lr = LinearRegression().fit(t, y)
lr_pred_dxy = lr.predict(t_future)

poly = make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(t, y)
poly_pred_dxy = poly.predict(t_future)

hw_model = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
hw_pred_dxy = hw_model.forecast(future_steps)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
axes[0].plot(series.index, series, color='black', linewidth=1, alpha=0.6, label='Observed')
axes[0].plot(future_dates, lr_pred_dxy, '--', color='royalblue', linewidth=2, label='Linear Regression')
axes[0].plot(future_dates, poly_pred_dxy, '--', color='darkorange', linewidth=2, label='Polynomial (deg 3)')
axes[0].plot(future_dates, hw_pred_dxy.values, '--', color='green', linewidth=2, label='Holt-Winters')
axes[0].axvline(series.index[-1], color='gray', linestyle=':', linewidth=1.5)
axes[0].set_title('Dollar Index — Full History + 36M Forecast')
axes[0].set_ylabel('Index Value')
axes[0].legend()

zoom_start = series.index[-1] - pd.DateOffset(years=5)
axes[1].plot(series[zoom_start:].index, series[zoom_start:], color='black', linewidth=1.5, alpha=0.7, label='Observed')
axes[1].plot(future_dates, lr_pred_dxy, '--', color='royalblue', linewidth=2, label='Linear Regression')
axes[1].plot(future_dates, poly_pred_dxy, '--', color='darkorange', linewidth=2, label='Polynomial (deg 3)')
axes[1].plot(future_dates, hw_pred_dxy.values, '--', color='green', linewidth=2, label='Holt-Winters')
axes[1].axvline(series.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
axes[1].set_title('Zoomed: Last 5 Years + 36M Forecast')
axes[1].set_ylabel('Index Value')
axes[1].legend()
plt.tight_layout()
plt.show()

---
# Part 5 — Multi-Factor Inflation Forecast (Ridge Regression)

Having understood each series individually in Parts 1–4, this section combines all four into a single predictive model for inflation expectations. The goal is to determine whether the *combination* of macro variables explains inflation better than any single series alone, and to generate a 36-month forecast that accounts for the joint trajectory of all predictors.

**Why Ridge Regression rather than OLS?**
Standard OLS regression is sensitive to multicollinearity — when predictors are correlated with each other, OLS assigns unstable, extreme coefficients that cancel out rather than reflecting genuine economic relationships. This dataset contains several highly correlated predictors by construction: housing prices and their lagged values move together, and the dollar index and its lags are near-identical at short horizons. Ridge regression addresses this by adding an L2 penalty that shrinks coefficients toward zero proportionally, producing stable and interpretable estimates even in the presence of correlated features.

Both OLS and Ridge are run in parallel so that the effect of the regularisation can be observed directly through the difference in their forecasts.

---

### 5.1 Load & Align All Series

All four series are loaded fresh using a standardised helper function and resampled to a common monthly frequency. The key alignment decisions are:

- **Housing (quarterly → monthly):** Forward-filled rather than interpolated. Forward-filling uses the most recently known value until the next observation is available, which is the correct approach for a quarterly release — the Q1 value is *known* in April, May, and June, so those months legitimately take the Q1 value. Linear interpolation would imply knowing future values, which is a form of look-ahead bias.
- **Dollar index (daily → monthly):** Averaged over the month. The monthly mean captures the central tendency of the exchange rate over the period without being sensitive to end-of-month anomalies.
- **Inner join:** Only months where all four series have valid observations are retained. This ensures no imputed or extrapolated values enter the regression, at the cost of a shorter usable sample.

In [ ]:
def load(path, col, resample='MS', agg='mean'):
    d = pd.read_csv(DATA_DIR + path, parse_dates=['observation_date'])
    d = d.rename(columns={'observation_date': 'date', d.columns[1]: col})
    d = d.set_index('date').sort_index()
    if resample:
        d = d.resample(resample).agg(agg)
    return d

inflation  = load('T10YIEM.csv',   'inflation')
fedfunds   = load('FEDFUNDS.csv',  'fedfunds')
housing    = load('MSPUS.csv',     'housing')
dxy        = load('DTWEXBGS.csv',  'dxy')

housing = housing.resample('MS').ffill()

df = inflation.join([fedfunds, housing, dxy], how='inner').dropna()
print(f'Combined dataset: {df.shape[0]} months | {df.index[0].date()} to {df.index[-1].date()}')
df.describe().round(2)

### 5.2 Correlation Matrix

The correlation matrix is examined before feature engineering because it reveals the raw linear relationships between the original series. This serves two purposes:

1. **Identifying the strongest predictors of inflation** — features with high absolute correlation with the target are likely to be the most important in the regression.
2. **Diagnosing multicollinearity among predictors** — pairs of predictors with very high inter-correlation (|r| > 0.8) will be difficult for OLS to disentangle, reinforcing the case for Ridge regularisation.

The correlation matrix is computed on raw levels rather than changes at this stage to get a sense of the unconditional relationships. The feature engineering step that follows will create change and lag variables to better capture the dynamic relationships.

In [ ]:
corr = df.corr()
labels = ['Inflation', 'Fed Funds', 'Housing Price', 'Dollar Index']

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_title('Correlation Matrix — All Features vs Inflation')
plt.tight_layout()
plt.show()

### 5.3 Feature Engineering

Raw levels alone are insufficient predictors of inflation because macro relationships operate with a lag — a Fed rate hike in January does not cool inflation immediately but typically over 6–18 months. Four sets of features are constructed from each predictor:

- **Lags (1, 3, 6, 12 months):** Capture the delayed transmission of each variable to inflation. The 12-month lag is particularly important as it captures annual cycles and the typical monetary policy transmission horizon.
- **12-month change (YoY):** Removes the level trend from each series and focuses on the *rate of change*, which is often more predictive of inflation than the absolute level. For example, it is the *increase* in oil prices, not their absolute level, that tends to cause inflationary pressure.

The `dropna()` call after feature construction removes the initial rows that cannot be populated due to the lagging and differencing operations. This is expected and does not represent data loss — it simply reflects the minimum history required to compute the 12-month lag.

In [ ]:
for col in ['fedfunds', 'housing', 'dxy']:
    for lag in [1, 3, 6, 12]:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)
    df[f'{col}_yoy'] = df[col].diff(12)

df.dropna(inplace=True)
print(f'Dataset after feature engineering: {df.shape[0]} months, {df.shape[1] - 1} features')

### 5.4 Pairwise Scatter: Each Feature vs Inflation

Pairwise scatter plots with OLS trend lines are examined before fitting the multivariate model to assess whether the relationships appear linear. A non-linear scatter (e.g. a clear curve or cluster structure) would suggest that a linear model may underfit that relationship, and that transformation or interaction terms might be needed.

The Pearson correlation coefficient (r) is displayed on each subplot for a quick quantitative summary of linearity and direction. These values should be interpreted alongside the scatter pattern — a high |r| with a curved scatter indicates a correlation driven by outliers or a non-linear relationship that the model will handle imperfectly.

In [ ]:
base_features = ['fedfunds', 'housing', 'dxy', 'fedfunds_yoy', 'housing_yoy', 'dxy_yoy']
labels_map = {'fedfunds': 'Fed Funds Rate', 'housing': 'Median Home Price',
              'dxy': 'Dollar Index', 'fedfunds_yoy': 'Fed Funds YoY Δ',
              'housing_yoy': 'Housing YoY Δ', 'dxy_yoy': 'DXY YoY Δ'}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, feat in enumerate(base_features):
    axes[i].scatter(df[feat], df['inflation'], alpha=0.4, s=12, color='steelblue')
    m, b = np.polyfit(df[feat], df['inflation'], 1)
    xr = np.linspace(df[feat].min(), df[feat].max(), 100)
    axes[i].plot(xr, m * xr + b, color='red', linewidth=1.5)
    r = df[['inflation', feat]].corr().iloc[0, 1]
    axes[i].set_title(f'{labels_map[feat]}  (r={r:.2f})')
    axes[i].set_ylabel('Inflation (%)')
plt.suptitle('Feature vs Inflation Expectations', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

### 5.5 Build & Evaluate Models

Models are evaluated using two complementary metrics:

- **In-sample R²:** Measures how well the model explains the variance in the training data. A high in-sample R² with a poor cross-validation score indicates overfitting.
- **Time-series cross-validation RMSE (5-fold):** The data is split into 5 sequential folds, each training on all prior data and testing on the next period. This is the correct validation approach for time series — random k-fold would allow future data to leak into the training set, producing optimistically biased estimates.

Both OLS and Ridge are evaluated so the effect of regularisation on generalisation can be measured directly. If Ridge has a meaningfully lower CV RMSE than OLS, it confirms that the regularisation is reducing overfitting caused by correlated features.

In [ ]:
feature_cols = [c for c in df.columns if c != 'inflation']
X = df[feature_cols]
y = df['inflation']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

tscv = TimeSeriesSplit(n_splits=5)

lr_model = LinearRegression()
lr_model.fit(X_scaled, y)
lr_cv = -cross_val_score(lr_model, X_scaled, y, cv=tscv, scoring='neg_mean_squared_error')

ridge = Ridge(alpha=1.0)
ridge.fit(X_scaled, y)
ridge_cv = -cross_val_score(ridge, X_scaled, y, cv=tscv, scoring='neg_mean_squared_error')

print('=== In-Sample ===')
print(f'OLS   R²: {r2_score(y, lr_model.predict(X_scaled)):.4f}  RMSE: {np.sqrt(mean_squared_error(y, lr_model.predict(X_scaled))):.4f}')
print(f'Ridge R²: {r2_score(y, ridge.predict(X_scaled)):.4f}  RMSE: {np.sqrt(mean_squared_error(y, ridge.predict(X_scaled))):.4f}')
print('\n=== Time-Series CV RMSE (5-fold) ===')
print(f'OLS   CV RMSE: {np.sqrt(lr_cv).mean():.4f} ± {np.sqrt(lr_cv).std():.4f}')
print(f'Ridge CV RMSE: {np.sqrt(ridge_cv).mean():.4f} ± {np.sqrt(ridge_cv).std():.4f}')

### 5.6 Ridge Feature Importance

Ridge coefficients on standardised features are used as a proxy for feature importance. Standardisation (subtracting the mean and dividing by standard deviation) is applied before fitting so that all features are on the same scale — without this, features measured in large units (e.g. housing prices in dollars) would appear to have negligible coefficients purely due to scale, not because they are unimportant.

The coefficient sign is economically meaningful: a positive coefficient on a feature means an increase in that feature is associated with higher inflation expectations, and vice versa. These signs should be checked against economic theory — for example, a higher Fed Funds Rate should be associated with *lower* inflation (negative coefficient on the rate level), while a positive coefficient on the YoY change in oil prices aligns with cost-push theory.

Features with coefficients near zero have been effectively regularised out by Ridge — they are providing little incremental information given the other features already in the model.

In [ ]:
coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': ridge.coef_})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=True).index)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['steelblue' if c >= 0 else 'tomato' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Ridge Regression Coefficients (Standardized)\nBlue = raises inflation, Red = lowers inflation')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

### 5.7 Actual vs Fitted

The actual vs fitted plot is the primary diagnostic for model fit quality. Key things to look for:

- **Systematic bias:** If the model consistently over- or under-predicts during a specific period (e.g. always too low during the 2022 inflation surge), this indicates the model is missing a structural driver during that regime.
- **Residual magnitude:** Large residuals in the bar chart indicate observations the model could not explain well. If large residuals cluster around specific events (recessions, policy shifts), this is expected — these are non-linear shocks that a linear model cannot fully capture.
- **OLS vs Ridge comparison:** If the two models produce very different fitted values, it signals that multicollinearity was significantly affecting the OLS coefficients. Convergence between the two suggests the features are not excessively collinear.

In [ ]:
df['fitted_ols']   = lr_model.predict(X_scaled)
df['fitted_ridge'] = ridge.predict(X_scaled)
df['residual']     = y - df['fitted_ridge']

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
axes[0].plot(df.index, df['inflation'],    color='black',      linewidth=1.2, label='Actual Inflation')
axes[0].plot(df.index, df['fitted_ols'],   color='royalblue',  linewidth=1,   linestyle='--', alpha=0.8, label='OLS Fitted')
axes[0].plot(df.index, df['fitted_ridge'], color='green',      linewidth=1,   linestyle='--', alpha=0.8, label='Ridge Fitted')
axes[0].set_title('Actual vs Fitted Inflation Expectations')
axes[0].set_ylabel('Inflation (%)')
axes[0].legend()

axes[1].bar(df.index, df['residual'],
            color=np.where(df['residual'] >= 0, 'steelblue', 'tomato'), width=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Ridge Residuals')
axes[1].set_ylabel('Error (pp)')
plt.tight_layout()
plt.show()

### 5.8 36-Month Inflation Forecast

The forecast is generated in two steps:

**Step 1 — Project each predictor forward:** Each raw feature (Fed Funds Rate, housing price, dollar index) is projected 36 months using its own individual linear trend. This is a deliberate simplification — more sophisticated approaches (e.g. VAR models, scenario analysis) would be appropriate for production forecasting. The linear trend projection is used here to produce a single coherent baseline scenario rather than requiring external assumptions about future predictor values.

**Step 2 — Feed projections into the trained model:** The projected predictor values, along with their lagged and YoY transformed versions reconstructed from the extended history, are passed through the fitted Ridge model to produce monthly inflation forecasts.

The **model agreement column** in the summary table flags months where OLS and Ridge forecasts diverge by more than 0.3 percentage points. Divergence is a useful signal: when the two models agree, the forecast is likely robust to the choice of regularisation. When they diverge, it indicates either that multicollinearity is distorting the OLS projection, or that the forecast is extrapolating into a range of predictor values not well-represented in the training data — both of which warrant caution in interpretation.

The zoomed 4-year panel allows the forecast trajectory to be assessed in the context of the most recent observed trend, which is the most relevant reference point for near-term projections.

In [ ]:
future_steps = 36
future_dates = pd.date_range(df.index[-1] + pd.DateOffset(months=1), periods=future_steps, freq='MS')

raw_features = ['fedfunds', 'housing', 'dxy']
future_raw = {}
t_hist = np.arange(len(df)).reshape(-1, 1)
t_fut  = np.arange(len(df), len(df) + future_steps).reshape(-1, 1)

for feat in raw_features:
    m = LinearRegression().fit(t_hist, df[feat].values)
    future_raw[feat] = m.predict(t_fut)

fut_df = pd.DataFrame(future_raw, index=future_dates)
extended = pd.concat([df[raw_features], fut_df])
for col in raw_features:
    for lag in [1, 3, 6, 12]:
        extended[f'{col}_lag{lag}'] = extended[col].shift(lag)
    extended[f'{col}_yoy'] = extended[col].diff(12)

X_future = extended.loc[future_dates, feature_cols].dropna()
valid_dates = X_future.index
X_future_scaled = scaler.transform(X_future)

forecast_ols   = lr_model.predict(X_future_scaled)
forecast_ridge = ridge.predict(X_future_scaled)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
axes[0].plot(df.index, df['inflation'], color='black', linewidth=1, alpha=0.6, label='Observed')
axes[0].plot(df.index, df['fitted_ridge'], color='gray', linewidth=0.8, linestyle='--', alpha=0.6, label='Ridge Fitted')
axes[0].plot(valid_dates, forecast_ols,   '--', color='royalblue', linewidth=2, label='OLS Forecast')
axes[0].plot(valid_dates, forecast_ridge, '--', color='green',     linewidth=2, label='Ridge Forecast')
axes[0].axvline(df.index[-1], color='gray', linestyle=':', linewidth=1.5)
axes[0].set_title('Multi-Factor Inflation Forecast — Full History')
axes[0].set_ylabel('Inflation Expectation (%)')
axes[0].legend()

zoom_start = df.index[-1] - pd.DateOffset(years=4)
axes[1].plot(df[zoom_start:].index, df.loc[zoom_start:, 'inflation'],
             color='black', linewidth=1.5, label='Observed')
axes[1].plot(df[zoom_start:].index, df.loc[zoom_start:, 'fitted_ridge'],
             color='gray', linewidth=1, linestyle='--', alpha=0.7, label='Ridge Fitted')
axes[1].plot(valid_dates, forecast_ols,   '--', color='royalblue', linewidth=2, label='OLS Forecast')
axes[1].plot(valid_dates, forecast_ridge, '--', color='green',     linewidth=2, label='Ridge Forecast')
axes[1].axvline(df.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
axes[1].set_title('Zoomed: Last 4 Years + 36M Forecast')
axes[1].set_ylabel('Inflation Expectation (%)')
axes[1].legend()
plt.tight_layout()
plt.show()

print('\n--- 36-Month Inflation Forecast Summary ---')
summary = pd.DataFrame({
    'Date':               valid_dates.strftime('%Y-%m'),
    'OLS Forecast (%)':   forecast_ols.round(3),
    'Ridge Forecast (%)': forecast_ridge.round(3)
})
print(summary.to_string(index=False))